# JSON-RPC MCP Testing - Simple Protocol Test

This notebook tests the MCP server using direct JSON-RPC calls to verify basic protocol functionality.

**Prerequisites:** Make sure your MCP server is running on `localhost:8005`

This is a simpler alternative to the full MCP client testing and is good for basic verification.

## Setup: Import Dependencies

Importing required libraries for HTTP requests and JSON handling.

In [9]:
import requests
import json

# Configuration
base_url = "http://localhost:8005/mcp"
headers = {"Content-Type": "application/json"}

print('✅ Libraries imported successfully')
print(f'🔗 Target server: {base_url}')
print('🚀 Ready to test JSON-RPC MCP protocol')

✅ Libraries imported successfully
🔗 Target server: http://localhost:8005/mcp
🚀 Ready to test JSON-RPC MCP protocol


## Test 1: Server Connectivity Check

Basic connectivity test to verify the server is responding.

In [10]:
# Basic connectivity test
print("🔌 Testing server connectivity...")

try:
    # Simple ping to check if server is responding
    response = requests.get("http://localhost:8005", timeout=5)
    print(f"✅ Server is responding (status: {response.status_code})")
    connectivity_status = True
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to server - is it running on port 8005?")
    connectivity_status = False
except Exception as e:
    print(f"⚠️  Connection issue: {e}")
    connectivity_status = False

if connectivity_status:
    print('✅ Connectivity test PASSED')
else:
    print('❌ Connectivity test FAILED - check if server is running')

🔌 Testing server connectivity...
✅ Server is responding (status: 404)
✅ Connectivity test PASSED


## Test 2: List Available Tools

Testing the `tools/list` JSON-RPC method to discover available tools.

In [11]:
# Test 1: List tools using JSON-RPC
print("📋 Testing tools/list JSON-RPC method...")

payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list"
}

try:
    response = requests.post(base_url, json=payload, headers=headers, timeout=10)
    print(f"HTTP Status: {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        print(f"Response structure: {list(data.keys())}")
        
        if "result" in data and "tools" in data["result"]:
            tools = data["result"]["tools"]
            print(f"✅ Found {len(tools)} tools:")
            
            for i, tool in enumerate(tools, 1):
                tool_name = tool.get('name', 'Unknown')
                tool_desc = tool.get('description', 'No description')
                print(f"  {i}. {tool_name}")
                print(f"     Description: {tool_desc[:80]}...")

            # Check if our search tool exists
            search_tool = next((t for t in tools if t.get('name') == 'search_interaction_events'), None)
            if search_tool:
                print("\n✅ Found search_interaction_events tool!")
                print(f"   Input schema keys: {list(search_tool.get('inputSchema', {}).get('properties', {}).keys())}")
                tools_list_result = True
            else:
                print("\n❌ search_interaction_events tool not found!")
                tools_list_result = False
        else:
            print("❌ Invalid response format - missing result/tools")
            print(f"Full response: {data}")
            tools_list_result = False
    else:
        print(f"❌ HTTP error: {response.status_code}")
        print(f"Response: {response.text}")
        tools_list_result = False

except Exception as e:
    print(f"❌ Tools list request failed: {e}")
    tools_list_result = False

if tools_list_result:
    print('\n✅ Tools listing test PASSED')
else:
    print('\n❌ Tools listing test FAILED')

📋 Testing tools/list JSON-RPC method...
HTTP Status: 406
❌ HTTP error: 406
Response: {"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Not Acceptable: Client must accept both application/json and text/event-stream"}}

❌ Tools listing test FAILED


## Test 3: Call Search Tool - Exact Match

Testing the `tools/call` method with exact event name matching.

In [12]:
# Test 2: Call search_interaction_events with exact match
print("🎯 Testing search_interaction_events - exact match...")

payload = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "search_interaction_events",
        "arguments": {
            "query": "wayfinder_start",
            "max_results": 3
        }
    }
}

try:
    response = requests.post(base_url, json=payload, headers=headers, timeout=15)
    print(f"HTTP Status: {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        
        if "result" in data:
            result = data["result"]
            print("✅ Tool call successful!")
            print(f"Result type: {type(result)}")
            
            # Check for content
            if isinstance(result, dict) and "content" in result:
                content = result["content"]
                if content and len(content) > 0:
                    first_content = content[0]
                    if isinstance(first_content, dict) and "text" in first_content:
                        response_text = first_content["text"]
                        print(f"Content preview: {response_text[:200]}...")
                        
                        # Check if we got expected results
                        if "wayfinder_start" in response_text:
                            print("✅ Found expected 'wayfinder_start' in response!")
                        
                        exact_match_result = True
                    else:
                        print(f"Content structure: {first_content}")
                        exact_match_result = True  # Still successful
                else:
                    print("⚠️  Empty content, but call succeeded")
                    exact_match_result = True
            else:
                print(f"Result structure: {list(result.keys()) if isinstance(result, dict) else type(result)}")
                exact_match_result = True  # Call succeeded even if format is different
        else:
            print("❌ No result in response")
            print(f"Response keys: {list(data.keys())}")
            print(f"Full response: {data}")
            exact_match_result = False
    else:
        print(f"❌ HTTP error: {response.status_code}")
        print(f"Response: {response.text}")
        exact_match_result = False

except Exception as e:
    print(f"❌ Exact match tool call failed: {e}")
    exact_match_result = False

if exact_match_result:
    print('\n✅ Exact match test PASSED')
else:
    print('\n❌ Exact match test FAILED')

🎯 Testing search_interaction_events - exact match...
HTTP Status: 406
❌ HTTP error: 406
Response: {"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Not Acceptable: Client must accept both application/json and text/event-stream"}}

❌ Exact match test FAILED


## Test 4: Call Search Tool - Fuzzy Search

Testing fuzzy search with partial keywords.

In [13]:
# Test 3: Call search_interaction_events with fuzzy search
print("🔍 Testing search_interaction_events - fuzzy search...")

payload = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "search_interaction_events",
        "arguments": {
            "query": "wayfinder",
            "max_results": 5
        }
    }
}

try:
    response = requests.post(base_url, json=payload, headers=headers, timeout=15)
    print(f"HTTP Status: {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        
        if "result" in data:
            result = data["result"]
            print("✅ Tool call successful!")
            
            # Extract and display content
            if isinstance(result, dict) and "content" in result:
                content = result["content"]
                if content and len(content) > 0:
                    response_text = content[0].get("text", "No text field")
                    print(f"Content preview: {response_text[:250]}...")
                    
                    # Check for wayfinder-related results
                    wayfinder_count = response_text.lower().count("wayfinder")
                    if wayfinder_count > 0:
                        print(f"✅ Found {wayfinder_count} wayfinder references in response!")
                    
                    fuzzy_search_result = True
                else:
                    print("⚠️  Empty content, but call succeeded")
                    fuzzy_search_result = True
            else:
                print("ℹ️  Different result format, but call succeeded")
                fuzzy_search_result = True
        else:
            print("❌ No result in response")
            fuzzy_search_result = False
    else:
        print(f"❌ HTTP error: {response.status_code}")
        fuzzy_search_result = False

except Exception as e:
    print(f"❌ Fuzzy search failed: {e}")
    fuzzy_search_result = False

if fuzzy_search_result:
    print('\n✅ Fuzzy search test PASSED')
else:
    print('\n❌ Fuzzy search test FAILED')

🔍 Testing search_interaction_events - fuzzy search...
HTTP Status: 406
❌ HTTP error: 406

❌ Fuzzy search test FAILED


## Test 5: Call Search Tool - Edge Cases

Testing edge cases like empty queries and error handling.

In [14]:
# Test 4: Edge cases
print("🧪 Testing edge cases...")

# Test empty query
print("\n1. Testing empty query...")
payload = {
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "name": "search_interaction_events",
        "arguments": {
            "query": "",
            "max_results": 5
        }
    }
}

try:
    response = requests.post(base_url, json=payload, headers=headers, timeout=10)
    if response.status_code == 200:
        data = response.json()
        if "result" in data:
            print("✅ Empty query handled gracefully")
            empty_query_result = True
        else:
            print("⚠️  Empty query returned no result (acceptable)")
            empty_query_result = True
    else:
        print(f"❌ Empty query failed: {response.status_code}")
        empty_query_result = False
except Exception as e:
    print(f"❌ Empty query test error: {e}")
    empty_query_result = False

# Test non-matching query
print("\n2. Testing non-matching query...")
payload["params"]["arguments"]["query"] = "xyz123nonexistent"
payload["id"] = 5

try:
    response = requests.post(base_url, json=payload, headers=headers, timeout=10)
    if response.status_code == 200:
        data = response.json()
        if "result" in data:
            print("✅ Non-matching query handled gracefully")
            non_match_result = True
        else:
            print("⚠️  Non-matching query returned no result (acceptable)")
            non_match_result = True
    else:
        print(f"❌ Non-matching query failed: {response.status_code}")
        non_match_result = False
except Exception as e:
    print(f"❌ Non-matching query test error: {e}")
    non_match_result = False

edge_cases_result = empty_query_result and non_match_result

if edge_cases_result:
    print('\n✅ Edge cases test PASSED')
else:
    print('\n❌ Edge cases test FAILED')

🧪 Testing edge cases...

1. Testing empty query...
❌ Empty query failed: 406

2. Testing non-matching query...
❌ Non-matching query failed: 406

❌ Edge cases test FAILED


## Final JSON-RPC Test Summary

Overall assessment of JSON-RPC MCP protocol functionality.

In [15]:
print('🏁 JSON-RPC MCP TESTING COMPLETE')
print('=' * 50)

# Collect all test results
test_results = [
    ("Server Connectivity", connectivity_status if 'connectivity_status' in locals() else False),
    ("Tools Listing", tools_list_result if 'tools_list_result' in locals() else False),
    ("Exact Match Search", exact_match_result if 'exact_match_result' in locals() else False),
    ("Fuzzy Search", fuzzy_search_result if 'fuzzy_search_result' in locals() else False),
    ("Edge Cases", edge_cases_result if 'edge_cases_result' in locals() else False)
]

# Display results
passed_tests = 0
for test_name, result in test_results:
    status = "✅ PASSED" if result else "❌ FAILED"
    print(f"{test_name:.<20} {status}")
    if result:
        passed_tests += 1

print(f"\nOverall: {passed_tests}/{len(test_results)} tests passed")

if passed_tests == len(test_results):
    print('\n🎉 ALL JSON-RPC TESTS PASSED!')
    print('✅ MCP server JSON-RPC protocol is working correctly')
    print('✅ search_interaction_events tool is fully functional')
    print('✅ Server handles edge cases gracefully')
    print('📝 QA APPROVED - Ready for production!')
elif passed_tests >= 3:
    print('\n⚠️  Core functionality is working')
    print('✅ Basic MCP protocol is functional')
    print('⚠️  Some minor issues - review failed tests')
else:
    print('\n❌ Major issues detected')
    print('🔧 Review server configuration and fix failing tests')

print('\n📋 Next Steps:')
if passed_tests >= 4:
    print('1. ✅ Tool has passed basic QA')
    print('2. 📝 Ready for documentation updates')
    print('3. 🚀 Can proceed with production deployment')
else:
    print('1. 🔧 Fix failing tests')
    print('2. 🔄 Re-run tests until all pass')
    print('3. 📝 Update documentation only after QA passes')

print('=' * 50)

🏁 JSON-RPC MCP TESTING COMPLETE
Server Connectivity. ✅ PASSED
Tools Listing....... ❌ FAILED
Exact Match Search.. ❌ FAILED
Fuzzy Search........ ❌ FAILED
Edge Cases.......... ❌ FAILED

Overall: 1/5 tests passed

❌ Major issues detected
🔧 Review server configuration and fix failing tests

📋 Next Steps:
1. 🔧 Fix failing tests
2. 🔄 Re-run tests until all pass
3. 📝 Update documentation only after QA passes
